# ReF-DIM Training Notebook

This notebook contains the complete training script for ReF-DIM model.

## Setup Instructions

1. Run the first cell to install all required dependencies
2. Configure training parameters in the configuration cell
3. Run all cells sequentially to start training


## Step 1: Install Required Packages

Install all necessary dependencies for training. Run this cell first.


In [ ]:
# Install required packages
# Check CUDA version first (if using GPU)
# !nvidia-smi

# Install PyTorch (choose appropriate version based on your CUDA version)
# For CUDA 11.8:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# For CUDA 12.1:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# For CPU only:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Install other required packages
# !pip install numpy pillow tqdm thop swanlab lpips

# Check installed packages
import sys
required_packages = ['torch', 'torchvision', 'numpy', 'PIL', 'tqdm', 'thop', 'swanlab', 'lpips']
missing_packages = []

for package in required_packages:
    try:
        if package == 'PIL':
            __import__('PIL')
        else:
            __import__(package)
        print(f"✓ {package} is installed")
    except ImportError:
        print(f"✗ {package} is NOT installed")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
    print("Please uncomment and run the pip install commands above.")
else:
    print("\n✅ All required packages are installed!")


## Step 2: Import Libraries


In [ ]:
import os
import random
import numpy as np

import swanlab
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from thop import profile

from dataset import DIMDataset
from model.DIM import DIM


## Step 3: Configuration

Set your training parameters here.


In [ ]:
# Training Configuration
class Config:
    # Dataset path - modify this to your dataset path
    data_path = r'path/to/dataset'  # Should contain 'input' and 'target' subfolders
    
    # Training parameters
    n_iter = 100  # Number of epochs
    batch_size = 8
    learning_rate = 1e-5
    
    # Model parameters
    model_range = 6  # Number of enhancement stages
    loss_weights = [1, 0.2]  # [L2 weight, Perceptual weight]
    
    # Output path
    result_path = r'snapshot'
    
    # Random seed for reproducibility
    seed = 123
    
    # SwanLab project name
    project_name = "DIM-SCI"
    dataset_name = "LOL-blur-selected"

config = Config()


## Step 4: Verify Configuration


In [ ]:
# Verify dataset path exists
if os.path.exists(config.data_path):
    input_folder = os.path.join(config.data_path, 'input')
    target_folder = os.path.join(config.data_path, 'target')
    
    if os.path.exists(input_folder) and os.path.exists(target_folder):
        print(f"✓ Dataset path is valid: {config.data_path}")
        print(f"  - Input folder: {input_folder}")
        print(f"  - Target folder: {target_folder}")
    else:
        print(f"✗ Dataset folder structure is incorrect!")
        print(f"  Expected subfolders: 'input' and 'target' in {config.data_path}")
else:
    print(f"✗ Dataset path does not exist: {config.data_path}")
    print(f"  Please update config.data_path in the configuration cell above")

# Verify output path
print(f"\nOutput directory: {config.result_path}")
print(f"Configuration summary:")
print(f"  - Epochs: {config.n_iter}")
print(f"  - Batch size: {config.batch_size}")
print(f"  - Learning rate: {config.learning_rate}")
print(f"  - Model range: {config.model_range}")
print(f"  - Loss weights: L2={config.loss_weights[0]}, Percep={config.loss_weights[1]}")


## Step 5: Set Random Seeds for Reproducibility


In [ ]:
# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For deterministic behavior (may slow down training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
print(f"Random seed set to {config.seed}")


## Step 5: Initialize Data Loader


In [ ]:
# Initialize dataset and data loader
def worker_init_fn(worker_id):
    """Set random seed for DataLoader workers"""
    np.random.seed(config.seed + worker_id)
    random.seed(config.seed + worker_id)

# Create dataset
data = DIMDataset(config.data_path)
print(f"Dataset loaded: {len(data)} samples")

# Create data loader
data_loader = DataLoader(
    data, 
    batch_size=config.batch_size, 
    shuffle=True, 
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    num_workers=4  # Adjust based on your system
)
print(f"DataLoader created with batch_size={config.batch_size}")


## Step 6: Initialize Model


In [ ]:
# Initialize model and move to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create model
net = DIM(c1=3, c_hidden=32, range=config.model_range, weights=config.loss_weights).to(device)
print(f"Model created with range={config.model_range}")

# Calculate FLOPs and parameters
dummy_input = torch.randn(1, 3, 224, 224).to(device)
flops, params = profile(net, inputs=(dummy_input,), verbose=False)
print(f"Model parameters: {params:,}")
print(f"Model FLOPs: {flops:,}")


## Step 7: Initialize SwanLab for Experiment Tracking


In [ ]:
# Initialize SwanLab for experiment tracking
run = swanlab.init(
    project=config.project_name,
    config={
        "learning_rate": config.learning_rate,
        "epochs": config.n_iter,
        "loss_weight": {"L2": config.loss_weights[0], "Percep": config.loss_weights[1]},
        "GPU": torch.cuda.current_device() if torch.cuda.is_available() else "cpu",
        "batch_size": config.batch_size,
        "dataset": config.dataset_name,
        "seed": config.seed,
        "flops": int(flops),
        "params": int(params),
        "model_range": config.model_range,
    },
)
print("SwanLab initialized")


## Step 8: Initialize Optimizer


In [ ]:
# Initialize optimizer
optimizer = torch.optim.Adam(net.parameters(), lr=config.learning_rate)
print(f"Optimizer initialized with learning_rate={config.learning_rate}")


## Step 9: Training Loop


In [ ]:
# Training loop
best_avg_loss = float('inf')
best_model_path = None

# Create output directory
os.makedirs(config.result_path, exist_ok=True)

print("Starting training...")
print(f"Total epochs: {config.n_iter}")
print(f"Total batches per epoch: {len(data_loader)}")

for t in range(config.n_iter):
    net.train()
    # Track losses for this epoch
    losses = []
    L2_Losses = []
    Percep_Losses = []

    # Training loop over batches
    for (_, batch) in tqdm(enumerate(data_loader), desc=f'epoch-{t + 1}', total=len(data_loader)):
        # Get input and target values
        x = batch[0].to(device)
        gt = batch[1].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass and compute loss
        loss, L2_Loss, Percep_Loss = net._loss(x, gt)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        nn.utils.clip_grad_norm_(net.parameters(), 5)
        
        # Update weights
        optimizer.step()

        # Record losses
        losses.append(loss.item())
        L2_Losses.append(L2_Loss.item())
        Percep_Losses.append(Percep_Loss.item())

    # Calculate average loss for this epoch
    avg_loss = np.mean(losses)
    avg_L2_loss = np.mean(L2_Losses)
    avg_Percep_loss = np.mean(Percep_Losses)
    
    # Log to SwanLab
    run.log({
        "Loss": avg_loss, 
        "L2_Loss": avg_L2_loss, 
        "Percep_Loss": avg_Percep_loss
    })
    
    print(f"Epoch {t + 1}/{config.n_iter} - Loss: {avg_loss:.6f}, L2: {avg_L2_loss:.6f}, Percep: {avg_Percep_loss:.6f}")

    # Save best model
    if avg_loss < best_avg_loss:
        best_avg_loss = avg_loss
        best_model_path = os.path.join(config.result_path, 'best.pth')
        torch.save(net.state_dict(), best_model_path)
        print(f'  -> Best model saved with avg loss: {best_avg_loss:.6f}')

    # Save checkpoint every 5 epochs
    if (t + 1) % 5 == 0:
        model_path = os.path.join(config.result_path, f'epoch_{t + 1}.pth')
        torch.save(net.state_dict(), model_path)
        print(f'  -> Checkpoint saved at epoch {t + 1}')

print("\nTraining completed!")
if best_model_path:
    print(f'Best model saved at: {best_model_path} with avg loss: {best_avg_loss:.6f}')
    
run.finish()
